# Guest cleanup


## Acc altern lassen



In [ ]:
update auth.users set 
    created_at = now() - interval '8 days', 
    last_sign_in_at = now() - interval '8 days' 
    where id = 'DEINE_TEST_UUID'; 

update auth.sessions 
    set updated_at = now() - interval '8 days' 
    where user_id = 'DEINE_TEST_UUID';

Wichtig: nur bei genau dieser einen Test-UUID ausführen, niemals ohne WHERE-Filter!

## Löschvorschau

In [ ]:
select * from public.preview_guest_cleanup()

Jetzt sollte das Testkonto mit category 'empty' (bei 90 Tagen 'withe data' ) und einem eligible_at in der Vergangenheit auftauchen.

### Wer wird bis zum nächsten Sonntag fällig

In [ ]:
select *
from public.preview_guest_cleanup()
where eligible_at <= date_trunc('week', now() + interval '1 day') + interval '3 hours 15 minutes'
order by eligible_at, user_id;

Das berechnet automatisch den nächsten Sonntag 03:15 UTC und zeigt dir alle Kandidaten, die bis dahin fällig werden.

## Scharfstellen 

In [ ]:
update public.guest_cleanup_settings set enabled=true, dry_run=false;

### Wann gelöscht wird
Der Cron-Job cleanup-guest-accounts ist fest eingeplant auf jeden Sonntag, 03:15 UTC (das ist 04:15 im Winter bzw. 05:15 im Sommer bei uns). Nur zu diesem Zeitpunkt ruft Postgres automatisch dispatch_guest_cleanup() auf, was wiederum deine Edge Function anstößt. Zwischen den Sonntagen passiert nichts von selbst – außer du löst es manuell mit select public.dispatch_guest_cleanup(); aus, so wie bei den Tests eben.

### Wer beim Lauf tatsächlich gelöscht wird
Bei jedem Lauf (ob Sonntag automatisch oder manuell ausgelöst) prüft die Function für jedes anonyme Gastkonto:

Kein Google verknüpft – sonst nie, egal wie inaktiv.
Keine laufende Konto-Übertragung (Gast→Google) – sonst übersprungen.
"Aktivität" = das späteste von: Kontoerstellung, letzter Login, letzte Sitzungsaktualisierung, letzte Dokumentänderung, letzter Historien-Eintrag, letzter Bild-Upload.
Ist dieses Konto ohne jegliche Bikes/Setups/Bilder → gilt es als "leer" → löschbar, sobald diese Aktivität 7 Tage zurückliegt (aktuell dein eingestellter Wert).
Hat es Daten (mind. ein Bike/Setup/Bild) → löschbar erst nach 90 Tagen Inaktivität.

Nur Konten, die diese Schwelle zum Zeitpunkt des Laufs schon überschritten haben, werden angefasst – alles andere bleibt unberührt, auch beim nächsten Sonntag wieder neu geprüft.

### Wie viele auf einmal
Pro Lauf maximal batch_size Konten (aktuell 20), und der Lauf bricht nach 120 Sekunden ab, falls mehr wären – die restlichen werden beim nächsten Lauf zuerst nachgeholt.

## Manuell löschen auslösen

In [ ]:
 select public.dispatch_guest_cleanup()

## Löschverlauf anzeigen 

In [ ]:
select * from public.guest_cleanup_log